# Phase 1.6 — P2, dynamics in deltas (tile 32UNU)

Two questions, in this order.

**(a) Gate K2 — the reconstruction floor.** Can a linear head read *current*
NDVI out of `E(frame_t)` at all? An encoder that cannot has no appearance
signal to build on, so any later result about it is uninterpretable. The
verdict is recorded per encoder in the results table; an encoder that fails is
marked `audited: lossy` and **excluded from P3**. That is a result, not a
failure.

**(b) The delta probe.** Do embedding *changes* track real NDVI change, or does
the encoder discard it?

Nothing here is fine-tuned. No weight is loaded and no embedding is recomputed:
this notebook reads the Phase 1.2 `.npz` cache, the cubes (for NDVI, through
`data.ndvi.ndvi` and nothing else) and the manifest. CPU only.

**Three things this phase will not let you get wrong.**

1. **The gap is measured in DAYS**, on `daily_axis_index` — never on
   `original_axis_index`, which counts *acquisitions* and is the embedding join
   key. On this subset the two disagree on every pair by about a factor of five.
   Step 8 asserts that, on the real data, before anything is fitted.
2. **Common-masking is mandatory.** A pair's NDVI change is computed over pixels
   valid in *both* frames. Differencing two per-frame means compares two
   different pieces of ground and calls the difference a change.
3. **The gap-length-alone control is not optional.** It is P2's analogue of P1's
   degenerate control, and on this subset it beats every encoder on the
   *magnitude* target. `margin_over_control`, not the raw correlation, is the
   number to quote.

## Step 1: Install, then restart

In [1]:
import importlib.util, os, IPython
SENTINEL = "/content/.phase1_6_installed"
try:
    import google.colab            # noqa: F401
    ON_COLAB = True
except ImportError:
    # find_spec("google.colab") is NOT equivalent: it raises rather than
    # returning None when the parent `google` package is absent.
    ON_COLAB = False

if not ON_COLAB:
    print("not on Colab: skipping the install and the restart.")
    print("Run the notebook against your own environment (pip install -r "
          "requirements.txt) and continue from Step 2.")
elif os.path.exists(SENTINEL):
    print("Already installed in this runtime, skipping.")
    print(f"(delete {SENTINEL} and re-run to force a reinstall)")
else:
    # Not -q. A pip resolution failure here is the likeliest cause of every
    # later failure, and -q hides it.
    !pip install earthnet s3fs xarray zarr netCDF4 scikit-learn scipy

    # torch arrives with Colab and is imported transitively by encoders/.
    if importlib.util.find_spec("torch") is None:
        !pip install torch

    import subprocess, sys
    probe = ("import s3fs, xarray, zarr, netCDF4, earthnet, pandas, numpy, "
             "torch, sklearn, scipy, joblib")
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(
            "Install did not take. Read the pip output above for the real "
            "conflict. Do not continue: Step 12 would fail with no estimator."
        )

    open(SENTINEL, "w").write("ok")
    print("\n" + "=" * 70)
    print("INSTALL VERIFIED. RESTARTING THE RUNTIME NOW. This is expected.")
    print("When it comes back, continue from Step 2. Do not re-run this cell.")
    print("=" * 70)
    IPython.get_ipython().kernel.do_shutdown(True)

not on Colab: skipping the install and the restart.
Run the notebook against your own environment (pip install -r requirements.txt) and continue from Step 2.


## Step 2: Bootstrap

In [2]:
import os, sys, glob, zipfile, textwrap

REQUIRED = ["data/ndvi.py", "data/loader.py", "data/paths.py",
            "data/climatology.py", "encoders/manifest.py",
            "encoders/pipeline.py", "probes/cv.py",
            "probes/p1_appearance.py", "probes/p2_deltas.py",
            "tests/test_cv_folds.py", "tests/test_p2_deltas.py",
            "tests/conftest.py"]
ZIP_NAME = "phase1_6_repo.zip"
PHASE = "phase1_6"
INPUT_PHASE = "phase1_2"          # resolved by the shared block; NEVER read here

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except ImportError:
    DRIVE = None
    print("not on Colab, assuming the repo is the current directory")

def looks_like_repo(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in REQUIRED)

REPO = None
if DRIVE:
    zips = glob.glob(f"{DRIVE}/**/{ZIP_NAME}", recursive=True)
    unzipped = [os.path.dirname(os.path.dirname(h))
                for d in ("*", "*/*", "*/*/*")
                for h in glob.glob(f"{DRIVE}/{d}/probes/cv.py")]
    unzipped = [d for d in unzipped if looks_like_repo(d)]

    if zips:
        REPO = os.path.dirname(zips[0])
        marker = os.path.join(REPO, "probes", "p2_deltas.py")
        # Re-extract when the zip is newer than what is on disk. Without this a
        # freshly uploaded zip is ignored because an old checkout sits next to
        # it, and you debug last week's code.
        stale = (not os.path.exists(marker)
                 or os.path.getmtime(zips[0]) > os.path.getmtime(marker))
        if stale:
            print(f"found {zips[0]}")
            print(f"extracting into {REPO} (zip is newer)")
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(REPO)
            print()
            print("=" * 70)
            print("THE NOTEBOOK FILE ON DISK WAS JUST REPLACED.")
            print("Colab is still showing the cells it opened. To pick up the")
            print("new ones: File > Open notebook > Google Drive, and open")
            print("   " + os.path.join(REPO, "notebooks"))
            print("Until you do, the .py files are new and these cells are old.")
            print("=" * 70)
        else:
            print(f"using existing checkout at {REPO} (zip is not newer)")
    elif unzipped:
        REPO = unzipped[0]
        print(f"found unzipped repo, no zip present: {REPO}")
else:
    # Off Colab, walk up from the working directory: running the notebook from
    # notebooks/ is normal and must not be mistaken for a missing checkout.
    d = os.getcwd()
    while not looks_like_repo(d) and os.path.dirname(d) != d:
        d = os.path.dirname(d)
    REPO = d

if not looks_like_repo(REPO):
    raise RuntimeError(textwrap.dedent(f"""
        Could not find the Phase 1.6 code.

        Fix, 2 minutes:
          1. Run make_zip.sh locally to build {ZIP_NAME}
          2. Open https://drive.google.com
          3. Make a NEW subfolder  My Drive / NeurIPS-CCAI-2026 / phase1_6
          4. Drag {ZIP_NAME} into it (do not unzip)
          5. Re-run this cell.

        One subfolder per phase is deliberate: deleting phase1_6/ removes
        everything Phase 1.6 created and nothing an earlier phase depends on.
        data/raw stays at the project root -- it is shared, not a phase.

        Searched under: {DRIVE}
        Needed all of: {REQUIRED}
        Resolved REPO = {REPO}
    """).strip())

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

from data.paths import RAW_DIR, describe_phase, phase_dir

# --- READ-ONLY inputs, resolved wherever they already live -----------------
# === RESOLVER (pinned by tests/test_notebook_resolver.py) -- BEGIN ===
# Extracted and exercised by that test against a simulated Drive tree, so
# the precedence rule below cannot silently regress into "first hit wins".
def _candidates(rel, pattern="*"):
    """Every directory on Drive that could be `rel`, with its file count.

    Searched: this checkout, then Drive one, two and three levels down. Three,
    because phases are subfolders of one project folder -- the Phase 1.2
    embeddings sit at
        MyDrive / NeurIPS-CCAI-2026 / phase1_2 / data/phase1_2/embeddings
    which is two wildcards, while the shared cubes at
        MyDrive / NeurIPS-CCAI-2026 / data/raw
    are one.
    """
    seen, out = set(), []
    cands = [os.path.join(REPO, rel)]
    if DRIVE:
        for depth in ("*", "*/*", "*/*/*"):
            cands += sorted(glob.glob(f"{DRIVE}/{depth}/{rel}"))
    for c in cands:
        c = os.path.abspath(c)
        if c in seen or not os.path.isdir(c):
            continue
        seen.add(c)
        out.append((c, len(glob.glob(os.path.join(c, pattern)))))
    return out


def _resolve(rel, pattern, label, foreign_phase=False):
    """Pick ONE directory, by evidence, and show every candidate considered.

    TAKING THE FIRST HIT IS NOT A SELECTION, and it cost a real run: a stale
    copy of data/phase1_2/embeddings sat INSIDE the phase1_3 checkout, the old
    "this checkout first" rule preferred it over the true Phase 1.2 folder, and
    the run died on a pre-schema file nobody knew was there.

    So: most files wins, and for ANOTHER phase's artefacts a directory inside
    THIS phase's checkout never beats one outside it, whatever the counts. That
    is the layout contract -- a phase reads its inputs in place and never owns
    a copy -- expressed as code rather than as a docstring.
    """
    cands = [(c, n) for c, n in _candidates(rel, pattern) if n > 0]
    if not cands:
        return os.path.join(REPO, rel), []        # the caller reports the gap
    repo_abs = os.path.abspath(REPO)

    def inside_repo(c):
        return os.path.commonpath([repo_abs, c]) == repo_abs

    # The penalty applies ONLY in a per-phase checkout. In a plain development
    # clone the repo root IS where data/phase1_2 belongs, so penalising "inside
    # the repo" there would be backwards -- and a warning that fires when
    # nothing is wrong is a warning nobody reads the second time.
    def demote(c):
        return foreign_phase and IS_PHASE_CHECKOUT and inside_repo(c)

    ranked = sorted(cands, key=lambda cn: (
        0 if demote(cn[0]) else -1,                           # outside first
        -cn[1],                                               # then the fullest
        len(cn[0]),                                           # then the shortest
    ))
    chosen = ranked[0][0]
    if len(cands) > 1:
        print(f"[resolve] {label}: {len(cands)} candidate directories hold files --")
        for c, n in ranked:
            mark = "  <- USING" if c == chosen else ""
            flag = "  [inside this checkout]" if inside_repo(c) else ""
            print(f"[resolve]     {n:>4} file(s)  {c}{flag}{mark}")
    if demote(chosen):
        print(f"[resolve] WARNING: {label} resolved INSIDE this phase's checkout:")
        print(f"[resolve]   {chosen}")
        print("[resolve] Another phase's artefacts do not belong here -- one phase")
        print("[resolve] reads another's in place and never owns a copy. This is")
        print("[resolve] almost certainly stale. Delete it and re-run Step 2 so")
        print("[resolve] the real directory is found.")
    return chosen, ranked


# Is this checkout a PHASE folder (Drive), or a plain clone (local dev)? The
# name settles it and covers both Drive layouts that have existed: the nested
# "NeurIPS-CCAI-2026/phase1_3" and the older sibling "…-2026-phase1_3".
IS_PHASE_CHECKOUT = PHASE in os.path.basename(os.path.abspath(REPO))

RAW, _raw_cands = _resolve(RAW_DIR, "*.nc", "RAW")
EMB_IN, _emb_cands = _resolve(os.path.join("data", INPUT_PHASE, "embeddings"),
                              "*.npz", "EMB_IN", foreign_phase=True)
os.makedirs(RAW, exist_ok=True)

# A phase checkout should not contain another phase's artefact tree at all,
# even an empty one: it shadows the real directory on every future run.
_intruder = os.path.join(REPO, "data", INPUT_PHASE)
if IS_PHASE_CHECKOUT and os.path.isdir(_intruder):
    print()
    print(f"[resolve] NOTE: {_intruder}")
    print(f"[resolve] exists inside the {PHASE} checkout. {INPUT_PHASE} "
          "artefacts belong in the")
    print(f"[resolve] {INPUT_PHASE} subfolder. Nothing here writes to it, but it "
          "will keep shadowing")
    print("[resolve] the real one until you delete it.")
# === RESOLVER -- END ===

# Phase-specific, and therefore OUTSIDE the fenced block: P2 additionally reads
# the per-pixel masks cached in 1.2b, which common-masking cannot be done
# without. Resolved by the same rule, from the same helper.
MASKS_IN, _mask_cands = _resolve(os.path.join("data", INPUT_PHASE, "masks"),
                                 "*.npz", "MASKS_IN", foreign_phase=True)

# --- this phase's OWN outputs ----------------------------------------------
RESULTS = phase_dir(PHASE, "results")

n_cubes = len(glob.glob(os.path.join(RAW, "*.nc")))
n_emb = len(glob.glob(os.path.join(EMB_IN, "*.npz")))
print(f"\nREPO    {REPO}")
print(f"RAW     {RAW}   ({n_cubes} cubes)"
      + ("" if n_cubes else "   <- Step 4 downloads them"))
print(f"EMB_IN  {EMB_IN}   ({n_emb} .npz)")
print(f"MASKS_IN {MASKS_IN}   "
      f"({len(glob.glob(os.path.join(MASKS_IN, '*.npz')))} .npz)")
print("        ^ P2 READS this cache: pooled + 4x4 grid embeddings, and the")
print("          per-pixel masks cached in 1.2b that common-masking needs.")
print("          It loads NO weight and recomputes NO embedding.")
print(f"RESULTS {RESULTS}   (this phase writes here only)")
describe_phase(PHASE)

from data.ndvi import ndvi
from encoders.manifest import build_manifest
from probes import cv
from probes import p1_appearance as p1
from probes import p2_deltas as p2
print(f"\nimports OK. canonical NDVI at {ndvi.__module__}, "
      f"splits at {cv.__name__}, modes {cv.MODES}")
print(f"P2 at {p2.__name__}: parts {p2.PARTS}")
print(f"         aggregations {p2.AGGREGATIONS}, delta targets {p2.DELTA_TARGETS}")
print(f"         fold modes {p2.FOLD_MODES}, feature levels {p2.FEATURE_LEVELS}")
print(f"         read-outs {p2.READOUTS}, model kinds {p2.MODEL_KINDS}")
print(f"         controls {p2.CONTROL_KINDS}, gap control degree "
      f"{p2.GAP_CONTROL_DEGREE}")
print(f"         K2 read at {p2.K2_PRIMARY}")
print(f"         band-matched baseline {p2.BAND_MATCHED_BASELINE!r}, "
      f"MI encoder {p2.MI_ENCODER!r} (never si_comparable)")
for f in REQUIRED:
    print(f"  ok  {f}")


# --- shell helper, defined here so it can never be skipped ------------------
# Named sh(), not run(): IPython has a %run magic. If a helper called run() is
# ever undefined, automagic silently rewrites run("...") into %run("...") and
# reports a confusing error about a missing script instead of a NameError.
import shlex, subprocess

PY = shlex.quote(sys.executable)

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")
    print(f"[exit 0] {cmd}")

print("helper ready: sh('<shell command>')")

not on Colab, assuming the repo is the current directory

REPO    /Users/benji/Code/NeurIPS CCAI 2026
RAW     /Users/benji/Code/NeurIPS CCAI 2026/data/raw   (20 cubes)
EMB_IN  /Users/benji/Code/NeurIPS CCAI 2026/data/phase1_2/embeddings   (100 .npz)
MASKS_IN /Users/benji/Code/NeurIPS CCAI 2026/data/phase1_2/masks   (20 .npz)
        ^ P2 READS this cache: pooled + 4x4 grid embeddings, and the
          per-pixel masks cached in 1.2b that common-masking needs.
          It loads NO weight and recomputes NO embedding.
RESULTS data/phase1_6/results   (this phase writes here only)
[paths] data/phase1_6: 3 file(s), 1.81 MB
[paths]   logs/: 1 file(s), 1.18 MB
[paths]   results/: 2 file(s), 0.62 MB



imports OK. canonical NDVI at data.ndvi, splits at probes.cv, modes ('cube', 'crossed', 'year', 'tile', 'spatial_block', 'temporal')
P2 at probes.p2_deltas: parts ('A_reconstruction', 'B_delta')
         aggregations ('cube_mean', 'cube_p90', 'cell_mean'), delta targets ('sign', 'magnitude')
         fold modes ('cube', 'loco', 'spatial_block'), feature levels ('grid_cell', 'pooled')
         read-outs ('norm', 'linear'), model kinds ('reconstruction', 'retention', 'delta', 'gap_only')
         controls ('retention', 'gap_only'), gap control degree 2
         K2 read at {'aggregation': 'cube_mean', 'feature_level': 'grid_cell', 'fold_mode': 'cube'}
         band-matched baseline 'raw_rgb_only', MI encoder 'satlas_s2_swinb_mi_rgb' (never si_comparable)
  ok  data/ndvi.py
  ok  data/loader.py
  ok  data/paths.py
  ok  data/climatology.py
  ok  encoders/manifest.py
  ok  encoders/pipeline.py
  ok  probes/cv.py
  ok  probes/p1_appearance.py
  ok  probes/p2_deltas.py
  ok  tests/test_cv_fo

## Step 3: Environment check

In [3]:
import glob, os, textwrap

import numpy as np, pandas as pd, sklearn, scipy, joblib
print(f"numpy {np.__version__} | pandas {pd.__version__} | "
      f"sklearn {sklearn.__version__} | scipy {scipy.__version__} | "
      f"joblib {joblib.__version__}")

N_JOBS = max(1, (os.cpu_count() or 2) - 1)
print(f"N_JOBS = {N_JOBS} (of {os.cpu_count()} CPUs). Wall-clock only.")

n = len(glob.glob(os.path.join(RAW, "*.nc")))
if n == 0:
    print(textwrap.dedent("""
        No cubes found. Step 4 downloads them (~15 s, 67 MB).
    """).strip())
else:
    print(f"{n} cubes at {RAW}")
print("\nThis phase READS the Phase 1.2 .npz cache (embeddings + per-pixel masks).\nIt loads NO model weight and recomputes NO embedding.")

numpy 2.0.2 | pandas 2.3.3 | sklearn 1.6.1 | scipy 1.13.1 | joblib 1.5.3
N_JOBS = 7 (of 8 CPUs). Wall-clock only.
20 cubes at /Users/benji/Code/NeurIPS CCAI 2026/data/raw

This phase READS the Phase 1.2 .npz cache (embeddings + per-pixel masks).
It loads NO model weight and recomputes NO embedding.


## Step 4: The cubes

In [4]:
if len(glob.glob(os.path.join(RAW, "*.nc"))) >= 20:
    print("cubes already present, skipping the download")
else:
    sh(f"{PY} -m data.download_greenearthnet --out {shlex.quote(RAW)} "
       f"--n 20 --tile 32UNU")
print(f"{len(glob.glob(os.path.join(RAW, '*.nc')))} cubes at {RAW}")

cubes already present, skipping the download
20 cubes at /Users/benji/Code/NeurIPS CCAI 2026/data/raw


## Step 5: Unit tests

The invariant is **0 failed**, not a particular pass count — a hard-coded count
goes stale every phase.

In [5]:
# pytest.ini already sets addopts = -q. Passing -q again makes it -qq,
# which hides the per-file progress.
sh(f"{PY} -m pytest tests")

$ '/Users/benji/Code/NeurIPS CCAI 2026/.venv/bin/python' -m pytest tests


........................................................................ [ 16%]


...................ssss............s.................................... [ 32%]


........................................................................ [ 48%]


........................................................................ [ 64%]


........................................................................ [ 80%]


........................................................................ [ 96%]
.................                                                        [100%]
=============================== warnings summary ===============================
tests/test_encoders.py::test_grid_landcover_aligns_with_the_embedding_grid
  <frozen importlib._bootstrap>:228: RuntimeWarning: numpy.ndarray size changed, may indicate binary incompatibility. Expected 16 from C header, got 96 from PyObject

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
444 passed, 5 skipped, 1 warning in 61.94s (0:01:01)


[exit 0] '/Users/benji/Code/NeurIPS CCAI 2026/.venv/bin/python' -m pytest tests


## Step 6: The REAL manifest, rebuilt from the cubes

**Rebuilt, never reused.** Phase 1.5 fixed two bugs in `encoders/manifest.py`
(E-OBS joined on the acquisition axis instead of the daily one; `year` read from
the filename's window-start rather than the frame's calendar year). Any manifest
cached before that is wrong in ways that are internally consistent.

P2 touches no weather, so bug (i) does not reach it directly — but the *axis
conflation behind it* is exactly the trap this phase's gap calculation has to
avoid, so the join is verified here anyway.

In [6]:
from data.loader import load_cube
from encoders.manifest import assert_strata_present, assert_weather_join

SAMPLES = [load_cube(p, verbose=False)
           for p in sorted(glob.glob(os.path.join(RAW, "*.nc")))]
MANIFEST = build_manifest(SAMPLES)
assert_strata_present(MANIFEST)

print()
JOIN = assert_weather_join(MANIFEST, RAW)
assert max(JOIN["max_abs_diff"].values()) == 0.0

off = (MANIFEST.daily_axis_index - MANIFEST.original_axis_index).to_numpy()
print(f"\nMANIFEST {MANIFEST.shape} | {MANIFEST.cube_id.nunique()} cubes | "
      f"tiles {sorted(MANIFEST.tile.unique())} | years {sorted(MANIFEST.year.unique())}")
print(f"the two axes differ on {int((off != 0).sum())}/{len(MANIFEST)} rows by "
      f"{off.min()}..{off.max()} steps (median {int(np.median(off))})")
assert "daily_axis_index" in MANIFEST.columns, "manifest predates the 1.5 fix"

[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


...

[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[manifest] 264 (cube, frame) rows over 20 cubes
[manifest] columns: ['cube_id', 'tile', 'year', 'timestamp', 'original_axis_index', 'daily_axis_index', 'day_of_year', 'pixel_bbox', 'clear_frac', 'landcover_stratum', 'landcover_dominant_frac', 'grid_landcover', 'grid_landcover_purity', 'grid_elevation_m', 'eobs_tg', 'eobs_fg', 'eobs_hu', 'eobs_pp', 'eobs_qq', 'eobs_rr', 'eobs_tn', 'eobs_tx']
[manifest] landcover strata (per cube): {'cropland': 8, 'grassland': 6, 'tree_cover': 6}
[manifest] landcover strata (per grid cell, 320 cells over 20 cubes): {'cropland': 127, 'tree_cover': 99, 'grassland': 88, 'built_up': 5, 'bare_sparse': 1}
[manifest] cubes whose cells are NOT all one class: 19/20 -- this is the within-cube stratum contrast the per-cube label was hiding
[manifest] in-cube E-OBS joined on daily_axis_index -- the day the frame was ACQUIRED (8): ['eobs_fg', 'eobs_hu', 'eobs_pp', 'eobs_qq', 'eobs_rr', 'eobs_tg', 'eobs_tn', 'eobs_tx']
[manifest] original_axis_index (acquisition axis)

[manifest] weather join VERIFIED against the cubes: 264 rows x 8 E-OBS variables over 20 cubes, max abs difference 0 (tolerance 1e-9)

MANIFEST (264, 22) | 20 cubes | tiles ['32UNU'] | years [np.int64(2018)]
the two axes differ on 264/264 rows by 4..122 steps (median 53)


## Step 7: The encoder cache, and the Part A target

Shapes printed and asserted for every array, per the project convention.

In [7]:
from encoders.pipeline import assert_embeddings_complete, audit_embeddings

CUBE_IDS = set(MANIFEST.cube_id)
AUDIT = audit_embeddings(EMB_IN, cube_ids=CUBE_IDS)
print()
assert_embeddings_complete(AUDIT, CUBE_IDS, p2.ENCODER_ORDER)

print()
ARRAYS = p2.encoder_arrays(MANIFEST, emb_dir=EMB_IN, verbose=True)

# The lookback covariate must actually VARY on the multi-image encoder and be
# exactly 0 on the single-image ones. All-zero is LEGITIMATE for four of the
# five caches, so no shape, dtype or finiteness check can tell a correct zero
# from a lost covariate -- and a lost one would silently halve the retention
# control while it kept printing full output.
p1.assert_window_span_days_informative(ARRAYS)

Y_FRAME = p2.frame_targets(MANIFEST, RAW, verbose=True)

for enc, a in ARRAYS.items():
    assert a["pooled"].shape[0] == len(MANIFEST), (enc, a["pooled"].shape)
    assert a["grid"].shape[:2] == (len(MANIFEST), 16), (enc, a["grid"].shape)
print(f"\nall five encoders aligned to the manifest's {len(MANIFEST)} rows")
print(f"Part A target: CURRENT NDVI, aggregations {p2.AGGREGATIONS}")
print(f"         cube_mean {Y_FRAME['cube_mean'].shape} "
      f"cell_mean {Y_FRAME['cell_mean'].shape}")

[audit] /Users/benji/Code/NeurIPS CCAI 2026/data/phase1_2/embeddings
[audit]   100 .npz on disk -> 100 usable (20 cubes x 5 encoders)
[audit]     dinov2_vitb14             20 cubes
[audit]     imagenet_vit_b16          20 cubes
[audit]     raw_features              20 cubes
[audit]     satlas_s2_swinb_mi_rgb    20 cubes
[audit]     satlas_s2_swinb_rgb       20 cubes

[audit] COMPLETE: all 20 x 5 = 100 (cube, encoder) pairs present at v3

[p2] raw_features             pooled (264, 35) | grid (264, 16, 35) | clear_frac (264,) | window_span_days (264,) (max 0 d)


[p2] imagenet_vit_b16         pooled (264, 1536) | grid (264, 16, 768) | clear_frac (264,) | window_span_days (264,) (max 0 d)
[p2] dinov2_vitb14            pooled (264, 3840) | grid (264, 16, 768) | clear_frac (264,) | window_span_days (264,) (max 0 d)


[p2] satlas_s2_swinb_rgb      pooled (264, 1024) | grid (264, 16, 1024) | clear_frac (264,) | window_span_days (264,) (max 0 d)
[p2] satlas_s2_swinb_mi_rgb   pooled (264, 1024) | grid (264, 16, 1024) | clear_frac (264,) | window_span_days (264,) (max 105 d)
[p1] window_span_days INFORMATIVE: satlas_s2_swinb_mi_rgb varies over 22 distinct values, 0-105 d; all single-image encoders exactly 0


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


...

[loader] dropping 120/150 timesteps with no acquisition


[loader] dropping 120/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[p2] frame targets: cube_mean (264,) | cube_p90 (264,) | cell_mean (264, 16) | 29/4224 cells fully masked (dropped, never filled)

all five encoders aligned to the manifest's 264 rows
Part A target: CURRENT NDVI, aggregations ('cube_mean', 'cube_p90', 'cell_mean')
         cube_mean (264,) cell_mean (264, 16)


## Step 8: The gap axis — the live check, on real data, before anything is fitted

`original_axis_index` counts **acquisitions** and is the embedding join key.
`daily_axis_index` counts **days**. Using the first as a gap is the same
category error that put P4's weather features on the wrong day — and it is
silent, because the wrong column is finite, integer, in range, monotone within
a cube, and correlated with the right one.

So the check is not a docstring. It asserts, on these 244 real pairs, that the
three candidate readings of "the gap" **materially disagree** — a check that
cannot tell the right column from the wrong one proves nothing.

In [8]:
PAIRS = p2.pair_index(MANIFEST, verbose=True)
AXES = p2.assert_gap_axes_disagree(PAIRS, verbose=True)

print()
print(f"  gap_days        (daily_axis_index)    median {AXES['median_gap_days']:.0f}"
      f"   <- THE gap")
print(f"  gap_acq_steps   (original_axis_index) median {AXES['median_gap_acq_steps']:.0f}"
      f"   <- the JOIN KEY. Never a gap.")
print(f"  frames between  (array position)      median {AXES['median_gap_frame_steps']:.0f}"
      f"   <- 1 by construction. Never a gap.")
print(f"  => {AXES['ratio_days_per_acq_step']:.1f} days per acquisition step; the two "
      f"axes agree on {AXES['n_axes_equal']}/{AXES['n_pairs']} pairs")
assert AXES["n_axes_equal"] == 0

[p2] pairs (244,) over 20 cubes (264 retained frames -> 244 consecutive pairs)
[p2]   gap_days (daily_axis_index): min 5 median 10 max 35 | distinct [5.0, 10.0, 15.0, 20.0, 25.0, 30.0, 35.0]
[p2] GAP AXIS CHECK on 244 real pairs: gap_days median 10 vs gap_acq_steps median 2 vs frames-between 1 -- 5.0 days per acquisition step, 0/244 pairs where the two axes agree

  gap_days        (daily_axis_index)    median 10   <- THE gap
  gap_acq_steps   (original_axis_index) median 2   <- the JOIN KEY. Never a gap.
  frames between  (array position)      median 1   <- 1 by construction. Never a gap.
  => 5.0 days per acquisition step; the two axes agree on 0/244 pairs


## Step 9: Common-masking, and the surviving pixel count per gap length

Restricted to pixels valid in **both** frames, from the per-pixel masks cached
in Phase 1.2b. If the intersection collapsed at some gap length that would be a
finding about what this benchmark can support — so it is reported, not silently
dropped.

In [9]:
p2.open_run_log()          # per-pair detail goes to the log, not to stdout
try:
    Y_PAIR = p2.build_pair_targets(PAIRS, MANIFEST, RAW,
                                mask_dir=MASKS_IN, verbose=True)
finally:
    p2.close_run_log()

print()
SURVIVAL = p2.summarise_pixel_survival(PAIRS, Y_PAIR, verbose=True)

assert Y_PAIR["cube_mean"].shape == (PAIRS.n_pairs,)
assert Y_PAIR["cell_mean"].shape == (PAIRS.n_pairs, 16)
print(f"\nper-pair detail: {p2.run_log_path()}")

[p2] per-pair detail -> data/phase1_6/logs/p2_run.log (stdout keeps shapes, K2 verdicts, the four controls and the headlines)


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


...

[loader] dropping 120/150 timesteps with no acquisition


[loader] dropping 120/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[p2] pair targets: cube_mean (244,) | cube_p90 (244,) | cell_mean (244, 16) | 0/244 pairs with ZERO common pixels | 58/3904 cells with none

[p2] COMMON-MASKED PIXEL SURVIVAL BY GAP LENGTH (16384 px per frame, 16 cells)
[p2]    gap_d  pairs   px_min   px_med  frac_min  frac_med  cells_min  zero
[p2]        5    106     4464    14668     0.272     0.895         12     0
[p2]       10     67     6761    13747     0.413     0.839         13     0
[p2]       15     31     6061    12821     0.370     0.783         14     0
[p2]       20     15     7023    14749     0.429     0.900         13     0
[p2]       25     14     8276    15612     0.505     0.953         13     0
[p2]       30     10    14219    16142     0.868     0.985         16     0
[p2]       35      1    13269    13269     0.810     0.810         16     0
[p2]   no collapse: 0 pairs of 244 have zero common pixels; the weakest gap is 15 d at median 0.783 surviving. Survi

## Step 10: Leakage is prevented by the signature — EXHIBIT, not gate

Both fitting functions take the training index set as a **required positional**
argument, so neither can be fitted on everything by omitting it. The gate is
`tests/test_p2_deltas.py`, which poisons in **both** directions: held-out poison
must not move the fit, and training poison must.

In [10]:
import inspect

for fn, idx in ((p2.fit_gap_control, "train_idx"),
                (p2.select_ridge_alpha, "train_rows")):
    sig = inspect.signature(fn)
    params = list(sig.parameters)
    print(f"{fn.__name__}{sig}".replace(", *,", ",\n     *,")[:200])
    assert params[2] == idx, params
    assert sig.parameters[idx].default is inspect.Parameter.empty
    assert not any(p.startswith("test") for p in params), params
    print(f"  ok  {idx} is positional #3 and has no default; no test argument exists")

# The exhibit: poison the held-out rows, then a training row.
# np.tile, not np.repeat: the training slice must cover at least degree+1
# distinct gap lengths or fit_gap_control correctly refuses it.
_g = np.tile([5.0, 10.0, 15.0, 20.0], 25)
_y = 0.001 * _g + np.sin(_g)
_tr = np.arange(50)
_before = p2.fit_gap_control(_g, _y, _tr)
_te_poison = _y.copy(); _te_poison[50:] += 10.0
_tr_poison = _y.copy(); _tr_poison[0] += 10.0
print(f"\ncoef                      {_before.coef}")
print(f"after poisoning TEST rows  {p2.fit_gap_control(_g, _te_poison, _tr).coef}"
      "   <- unchanged")
print(f"after poisoning a TRAIN row{p2.fit_gap_control(_g, _tr_poison, _tr).coef}"
      "   <- MOVED (this is what makes the line above mean something)")

fit_gap_control(gap_days, values, train_idx,
     *, degree: 'int' = 2, label: 'str' = '', verbose: 'bool' = False) -> 'GapControl'
  ok  train_idx is positional #3 and has no default; no test argument exists
select_ridge_alpha(block: 'FeatureBlock', manifest, train_rows, metric: 'str', group_of_row=None, inner_k: 'int' = 3, alphas: 'Sequence[float]' = (0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0), ve
  ok  train_rows is positional #3 and has no default; no test argument exists

coef                      [-1.84691562e+00  1.68415560e-01 -1.24868836e-03]
after poisoning TEST rows  [-1.84691562e+00  1.68415560e-01 -1.24868836e-03]   <- unchanged
after poisoning a TRAIN row[-0.10230023 -0.0731229   0.0065667 ]   <- MOVED (this is what makes the line above mean something)


## Step 11: Fold disjointness, and the pseudo-replicate structure

Effective n is **cubes**. 264 frames, 244 pairs and 4224 cells are not
independent observations: 16 cells share a sky, ~13 frames share a place.

In [11]:
cubes = MANIFEST.cube_id.to_numpy()
for mode in p2.FOLD_MODES:
    folds = p2.outer_folds(MANIFEST, mode, k=5, verbose=False)
    for tr, te in folds:
        assert not (set(cubes[tr]) & set(cubes[te])), f"{mode}: cube on both sides"
        a_tr, b_tr = np.isin(PAIRS.row_a, tr), np.isin(PAIRS.row_b, tr)
        assert (a_tr == b_tr).all(), f"{mode}: a pair straddles the split"
    print(f"  ok  {mode:<14} {len(folds):>2} folds, no cube and no PAIR on both sides")

print(f"\nrows {len(MANIFEST)} frames | {PAIRS.n_pairs} pairs | "
      f"{len(MANIFEST) * 16} cells")
print(f"effective n {MANIFEST.cube_id.nunique()} CUBES -- every interval below is "
      "clustered at that level")

  ok  cube            5 folds, no cube and no PAIR on both sides
  ok  loco           20 folds, no cube and no PAIR on both sides
  ok  spatial_block   5 folds, no cube and no PAIR on both sides

rows 264 frames | 244 pairs | 4224 cells
effective n 20 CUBES -- every interval below is clustered at that level


## Step 12: The run

Full per-pair output goes to `data/phase1_6/logs/p2_run.log`. Stdout keeps the
shape assertions, the K2 verdicts, the four control values and the headline
correlations.

In [12]:
import time

t0 = time.time()
RESULTS_DF = p2.run_p2(MANIFEST, RAW, emb_dir=EMB_IN,
                       mask_dir=MASKS_IN, verbose=True)
RESULTS_DF = p2.add_margins(RESULTS_DF)
RESULTS_DF = p2.add_k2_verdicts(RESULTS_DF)
print(f"\n[p2] {RESULTS_DF.shape[0]} rows x {RESULTS_DF.shape[1]} columns "
      f"in {time.time() - t0:.0f}s")

[p2] per-pair detail -> data/phase1_6/logs/p2_run.log (stdout keeps shapes, K2 verdicts, the four controls and the headlines)
[p2] manifest (264, 22) | 20 cubes | tiles ['32UNU'] | years [np.int64(2018)]
[p2] raw_features             pooled (264, 35) | grid (264, 16, 35) | clear_frac (264,) | window_span_days (264,) (max 0 d)


[p2] imagenet_vit_b16         pooled (264, 1536) | grid (264, 16, 768) | clear_frac (264,) | window_span_days (264,) (max 0 d)
[p2] dinov2_vitb14            pooled (264, 3840) | grid (264, 16, 768) | clear_frac (264,) | window_span_days (264,) (max 0 d)


[p2] satlas_s2_swinb_rgb      pooled (264, 1024) | grid (264, 16, 1024) | clear_frac (264,) | window_span_days (264,) (max 0 d)


[p2] satlas_s2_swinb_mi_rgb   pooled (264, 1024) | grid (264, 16, 1024) | clear_frac (264,) | window_span_days (264,) (max 105 d)


[loader] dropping 121/150 timesteps with no acquisition


...

[p2]   linear  imagenet_vit_b16         embedding     D=768   rho -0.101 [-0.210, +0.007]


[p2]   linear  dinov2_vitb14            embedding     D=768   rho -0.079 [-0.222, +0.065]


[p2]   linear  satlas_s2_swinb_rgb      embedding     D=1024  rho -0.064 [-0.250, +0.121]


[p2]   linear  satlas_s2_swinb_mi_rgb   embedding     D=1024  rho -0.013 [-0.111, +0.084]  [si_comparable=False]
[p2]   linear  raw_features             raw_rgb_only  D=21    rho -0.064 [-0.138, +0.011]


[p2]   linear  raw_features             raw_nir_ndvi  D=14    rho -0.003 [-0.116, +0.111]
[p2]   linear  none                     gap_days      D=3     rho +0.198 [-0.050, +0.446]   <- CONTROL (gap length alone, no embedding)
[p2] margins attached. 140/300 delta rows are AT OR BELOW the gap-length-alone control -- i.e. the embedding change adds nothing there that the calendar did not already carry

[p2] ====================================================================
[p2] GATE K2 -- reconstruction floor at cube_mean/grid_cell/cube, cube-clustered
[p2]   raw_features   R2 +0.440  <- the SPECIFIED baseline. It holds NDVI_mean..NDVI_p90,
[p2]                              so at the matched level it IS the target. See the docstring.
[p2]   raw_rgb_only   R2 +0.417  <- BAND-MATCHED (B02/B03/B04 only). The fair floor,
[p2]                              and the verdict that decides P3 inclusion.
[p2]   encoder                        R2                 CI    vs raw    paired CI on vs-raw   v

## Step 13: The table's own invariants

Every assertion the exit test names. A control that silently differs between
filtered views means the CSV was built inconsistently, so the identity is
asserted rather than trusted.

In [13]:
RANKING = p2.structural_hypothesis(RESULTS_DF, verbose=False)

p2.assert_results_complete(RESULTS_DF)
p2.assert_k2_verdict_recorded(RESULTS_DF)
p2.assert_control_identical_across_views(RESULTS_DF)
p2.assert_degenerate_control_present(RESULTS_DF)
p2.assert_mi_flagged_and_excluded(RESULTS_DF, RANKING)
p2.assert_effective_n_counts_cubes(RESULTS_DF)
for name in ("assert_results_complete", "assert_k2_verdict_recorded",
             "assert_control_identical_across_views",
             "assert_degenerate_control_present",
             "assert_mi_flagged_and_excluded",
             "assert_effective_n_counts_cubes"):
    print(f"  ok  {name}")

  ok  assert_results_complete
  ok  assert_k2_verdict_recorded
  ok  assert_control_identical_across_views
  ok  assert_degenerate_control_present
  ok  assert_mi_flagged_and_excluded
  ok  assert_effective_n_counts_cubes


## Step 14: Gate K2, the four controls, and the headline

In [14]:
p2.print_k2_verdicts(RESULTS_DF)
p2.print_controls(RESULTS_DF)
p2.print_headlines(RESULTS_DF)
_ = p2.structural_hypothesis(RESULTS_DF, verbose=True)


[p2] ====================================================================
[p2] GATE K2 -- reconstruction floor at cube_mean/grid_cell/cube, cube-clustered
[p2]   raw_features   R2 +0.440  <- the SPECIFIED baseline. It holds NDVI_mean..NDVI_p90,
[p2]                              so at the matched level it IS the target. See the docstring.
[p2]   raw_rgb_only   R2 +0.417  <- BAND-MATCHED (B02/B03/B04 only). The fair floor,
[p2]                              and the verdict that decides P3 inclusion.
[p2]   encoder                        R2                 CI    vs raw    paired CI on vs-raw   vs band  verdict / verdict(band)
[p2]   satlas_s2_swinb_rgb        +0.545 [ +0.462, +0.628]   +0.105 [ -0.118, +0.327]   +0.128  passed / passed  (not separable)
[p2]   imagenet_vit_b16           +0.521 [ +0.324, +0.718]   +0.081 [ -0.061, +0.223]   +0.105  passed / passed  (not separable)
[p2]   raw_features               +0.440 [ +0.169, +0.711]   +0.000 [ +0.000, +0.000]   +0.024  baseline / base

## Step 15: Robustness — the same ordering under every fold mode?

`spatial_block` killed everything in P1 and P4. Reported, not dropped.

In [15]:
p = p2.STRUCTURAL_PRIMARY
view = RESULTS_DF[(RESULTS_DF.part == "B_delta")
                  & (RESULTS_DF.aggregation == p["aggregation"])
                  & (RESULTS_DF.delta_target == p["delta_target"])
                  & (RESULTS_DF.feature_level == p["feature_level"])
                  & (RESULTS_DF.readout == p["readout"])
                  & (RESULTS_DF.feature_set == "embedding")]
print(view.pivot_table(index="encoder", columns="fold_mode",
                       values="margin_over_control").round(3).to_string())
print("\n(margin over the gap-length-alone control; the control itself:)")
ctrl = RESULTS_DF[(RESULTS_DF.model_kind == "gap_only")
                  & (RESULTS_DF.aggregation == p["aggregation"])
                  & (RESULTS_DF.delta_target == p["delta_target"])
                  & (RESULTS_DF.feature_level == p["feature_level"])
                  & (RESULTS_DF.readout == p["readout"])]
print(ctrl.set_index("fold_mode").score_mean.round(3).to_string())

fold_mode                cube   loco  spatial_block
encoder                                            
dinov2_vitb14           0.653  0.441          0.464
imagenet_vit_b16        0.575  0.410          0.358
raw_features            0.918  0.747          0.772
satlas_s2_swinb_mi_rgb  0.281  0.224          0.074
satlas_s2_swinb_rgb     0.598  0.492          0.480

(margin over the gap-length-alone control; the control itself:)
fold_mode
cube            -0.118
loco             0.039
spatial_block    0.009


## Step 16: Save, and list what this phase wrote

In [16]:
CSV = os.path.join(RESULTS, "p2_deltas_results.csv")
RESULTS_DF.to_csv(CSV, index=False)
SURVIVAL.to_csv(os.path.join(RESULTS, "p2_pixel_survival_by_gap.csv"), index=False)

back = pd.read_csv(CSV)
assert back.shape == RESULTS_DF.shape, (back.shape, RESULTS_DF.shape)
p2.assert_results_complete(back)
p2.assert_k2_verdict_recorded(back)
p2.assert_control_identical_across_views(back)
p2.assert_degenerate_control_present(back)
p2.assert_effective_n_counts_cubes(back)
print(f"wrote {CSV}")
print(f"  {back.shape[0]} rows x {back.shape[1]} columns, "
      f"{os.path.getsize(CSV) / 1e3:.0f} kB, re-read and re-validated")
print()
describe_phase(PHASE)
print()
print("K2 verdicts carried on every row:")
print(back.groupby("encoder")[["k2_verdict", "k2_verdict_band_matched"]]
      .first().to_string())

wrote data/phase1_6/results/p2_deltas_results.csv
  600 rows x 64 columns, 623 kB, re-read and re-validated

[paths] data/phase1_6: 3 file(s), 1.81 MB
[paths]   logs/: 1 file(s), 1.18 MB
[paths]   results/: 2 file(s), 0.62 MB

K2 verdicts carried on every row:
                            k2_verdict k2_verdict_band_matched
encoder                                                       
dinov2_vitb14           audited: lossy                  passed
imagenet_vit_b16                passed                  passed
none                     n/a (control)           n/a (control)
raw_features                  baseline                baseline
satlas_s2_swinb_mi_rgb  audited: lossy          audited: lossy
satlas_s2_swinb_rgb             passed                  passed


## Phase 1.6 is done when

- [ ] Step 5 reports **0 failed** (the count grows every phase; the count is not
      the invariant).
- [ ] Step 8 asserts, on the real 244 pairs, that `gap_days` and
      `original_axis_index` **disagree** — 0 pairs where the two axes agree.
- [ ] Step 9 reports the common-masked pixel count **per gap length**, including
      how many pairs (if any) collapse to zero.
- [ ] Step 13 passes all six table invariants, including the control being
      digit-for-digit identical across every filtered view.
- [ ] Step 14 records a **K2 verdict for all five encoders**, and names which
      are excluded from P3 — excluding only where the verdict is *separable*
      on the paired per-fold difference. "Did not beat the baseline" and "is
      measurably worse than it" are different findings at 20 cubes.
- [ ] Every headline carries a cube-clustered interval and an effective n in
      **cubes**.
- [ ] The multi-image encoder is reported, flagged `si_comparable=False`, and
      absent from the structural-hypothesis ranking.
- [ ] Step 16 re-reads the CSV from disk and re-validates it.
- [ ] Verbatim stdout archived to `notebooks/runs/`.